[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/14_kv_cache.ipynb)

# 🔴 Hard: KV Cache Attention

Implement **multi-head attention with KV caching** for efficient autoregressive generation.

During LLM inference, recomputing all key/value projections at every step is wasteful.
A **KV cache** stores previously computed K and V tensors so only the new token(s) need projection.

### Signature
```python
class KVCacheAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor, cache=None) -> tuple[torch.Tensor, tuple]:
        # x: (B, S_new, D) — new tokens
        # cache: None or (K_past, V_past) each (B, num_heads, S_past, d_k)
        # Returns: (output, (K_all, V_all))
```

### Requirements
- Inherit from `nn.Module`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` projections
- When `cache=None` (prefill): apply **causal mask**, return all K/V as cache
- When `cache` provided (decode): concat new K/V with cached, no causal mask needed for single-token decode
- Incremental decode must produce **identical** results to full forward pass

### Key Idea
```
Prefill:  [t0 t1 t2 t3] → full causal attention → cache = (K_{0:3}, V_{0:3})
Decode:   [t4]           → Q=t4, K/V=cache+t4  → cache = (K_{0:4}, V_{0:4})
Decode:   [t5]           → Q=t5, K/V=cache+t5  → cache = (K_{0:5}, V_{0:5})
```

### My notes:
#### Why not need to cache Q?

We cache K and V because the current/new query needs to attend to previous tokens' K/V.

We don't cache Q because once we've computed the attention output for a token, its query is never needed again during normal autoregressive decoding.


In [2]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [3]:
import torch
import torch.nn as nn
import math

In [38]:
# ✏️ YOUR IMPLEMENTATION HERE

class KVCacheAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        # pass  # Initialize W_q, W_k, W_v, W_o
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, cache=None):
        # 1. Project Q, K, V from x
        # 2. Reshape to multi-head: (B, num_heads, S, d_k)
        # 3. If cache exists, concat new K/V with cached K/V
        # 4. Compute attention (causal mask needed during prefill)
        # 5. Return (output, (K_all, V_all))
        # pass
        
        batch_size, seq_len_q = x.size(0), x.size(1)

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        print(Q.shape, K.shape, V.shape)
        
        # [Batch, Seq_len, d_model] -> [Batch, Seq_len, num_heads, head_dim]
        Q_split = Q.view(batch_size, seq_len_q, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        K_split = K.view(batch_size, seq_len_q, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        V_split = V.view(batch_size, seq_len_q, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        
        print(Q_split.shape, K_split.shape, V_split.shape)
        
        if cache is None:
            causal_mask = torch.triu(
                torch.ones(seq_len_q, seq_len_q, dtype=torch.bool, device=Q.device),
                diagonal=1,
            ) # -inf mask

            qk = (Q_split @ K_split.transpose(-2, -1) / math.sqrt(self.head_dim)).masked_fill(causal_mask, float('-inf'))
            multi_head_attn = torch.softmax(qk, dim=-1) @ V_split
        else:
            K_cache, V_cache = cache

            K_split = torch.cat([K_cache, K_split], dim=2) # concat along the sequence length dimension
            V_split = torch.cat([V_cache, V_split], dim=2) # concat along the sequence length dimension
            print("After concatenation:", K_split.shape, V_split.shape)
            
            if seq_len_q > 1:
                causal_mask = torch.triu(
                    torch.ones(seq_len_q, K_split.size(2), dtype=torch.bool, device=Q.device),
                    # Unlike prefill, K includes cached past tokens.
                    diagonal=K_cache.size(2) + 1,
                    # Shift the causal triangle right by cache_len so all cached K/V
                    # are visible, while each new query cannot see future tokens
                    # within the current incremental chunk.
                ) # -inf mask
                print("Causal mask shape:", causal_mask.shape)
                qk = (Q_split @ K_split.transpose(-2, -1) / math.sqrt(self.head_dim)).masked_fill(causal_mask, float('-inf'))
                multi_head_attn = torch.softmax(qk, dim=-1) @ V_split
            else:
                multi_head_attn = torch.softmax(Q_split @ K_split.transpose(-2, -1) / math.sqrt(self.head_dim), dim=-1) @ V_split

        
        print(multi_head_attn.shape) # the seq length of the output is the same as the query sequence length
        attn = multi_head_attn.permute(0, 2, 1, 3).contiguous().view(batch_size, seq_len_q, self.d_model) # concatenate the heads
        print(attn.shape)
        
        return self.W_o(attn), (K_split, V_split)

### Increment one-by-one:

In [40]:
# 🧪 Debug
torch.manual_seed(0)
attn = KVCacheAttention(d_model=64, num_heads=4)
x = torch.randn(1, 6, 64)

# Full forward
full_out, _ = attn(x)
print("Full output shape:", full_out.shape)  # (1, 6, 64)

# Incremental: prefill 4, decode 1, decode 1
out1, cache = attn(x[:, :4])
print("Cache K shape:", cache[0].shape)  # (1, 4, 4, 16)
out2, cache = attn(x[:, 4:5], cache=cache)
out3, cache = attn(x[:, 5:6], cache=cache)
inc_out = torch.cat([out1, out2, out3], dim=1)
print("Match:", torch.allclose(full_out, inc_out, atol=1e-5))

torch.Size([1, 6, 64]) torch.Size([1, 6, 64]) torch.Size([1, 6, 64])
torch.Size([1, 4, 6, 16]) torch.Size([1, 4, 6, 16]) torch.Size([1, 4, 6, 16])
torch.Size([1, 4, 6, 16])
torch.Size([1, 6, 64])
Full output shape: torch.Size([1, 6, 64])
torch.Size([1, 4, 64]) torch.Size([1, 4, 64]) torch.Size([1, 4, 64])
torch.Size([1, 4, 4, 16]) torch.Size([1, 4, 4, 16]) torch.Size([1, 4, 4, 16])
torch.Size([1, 4, 4, 16])
torch.Size([1, 4, 64])
Cache K shape: torch.Size([1, 4, 4, 16])
torch.Size([1, 1, 64]) torch.Size([1, 1, 64]) torch.Size([1, 1, 64])
torch.Size([1, 4, 1, 16]) torch.Size([1, 4, 1, 16]) torch.Size([1, 4, 1, 16])
After concatenation: torch.Size([1, 4, 5, 16]) torch.Size([1, 4, 5, 16])
torch.Size([1, 4, 1, 16])
torch.Size([1, 1, 64])
torch.Size([1, 1, 64]) torch.Size([1, 1, 64]) torch.Size([1, 1, 64])
torch.Size([1, 4, 1, 16]) torch.Size([1, 4, 1, 16]) torch.Size([1, 4, 1, 16])
After concatenation: torch.Size([1, 4, 6, 16]) torch.Size([1, 4, 6, 16])
torch.Size([1, 4, 1, 16])
torch.Size

### Increment by multiple should also match:

In [44]:
# 🧪 Debug
torch.manual_seed(0)
attn = KVCacheAttention(d_model=64, num_heads=4)
x = torch.randn(1, 15, 64)

# Full forward
full_out, _ = attn(x)
print("Full output shape:", full_out.shape)  # (1, 6, 64)

# Incremental: prefill 4, decode 6 then decode 5
out1, cache = attn(x[:, :4])
print("Cache K shape:", cache[0].shape)  # (1, 4, 4, 16)
out2, cache = attn(x[:, 4:10], cache=cache)
out3, cache = attn(x[:, 10:15], cache=cache)
inc_out = torch.cat([out1, out2, out3], dim=1)
print("Match:", torch.allclose(full_out, inc_out, atol=1e-5))

torch.Size([1, 15, 64]) torch.Size([1, 15, 64]) torch.Size([1, 15, 64])
torch.Size([1, 4, 15, 16]) torch.Size([1, 4, 15, 16]) torch.Size([1, 4, 15, 16])
torch.Size([1, 4, 15, 16])
torch.Size([1, 15, 64])
Full output shape: torch.Size([1, 15, 64])
torch.Size([1, 4, 64]) torch.Size([1, 4, 64]) torch.Size([1, 4, 64])
torch.Size([1, 4, 4, 16]) torch.Size([1, 4, 4, 16]) torch.Size([1, 4, 4, 16])
torch.Size([1, 4, 4, 16])
torch.Size([1, 4, 64])
Cache K shape: torch.Size([1, 4, 4, 16])
torch.Size([1, 6, 64]) torch.Size([1, 6, 64]) torch.Size([1, 6, 64])
torch.Size([1, 4, 6, 16]) torch.Size([1, 4, 6, 16]) torch.Size([1, 4, 6, 16])
After concatenation: torch.Size([1, 4, 10, 16]) torch.Size([1, 4, 10, 16])
Causal mask shape: torch.Size([6, 10])
torch.Size([1, 4, 6, 16])
torch.Size([1, 6, 64])
torch.Size([1, 5, 64]) torch.Size([1, 5, 64]) torch.Size([1, 5, 64])
torch.Size([1, 4, 5, 16]) torch.Size([1, 4, 5, 16]) torch.Size([1, 4, 5, 16])
After concatenation: torch.Size([1, 4, 15, 16]) torch.Size(

In [45]:
# ✅ SUBMIT
from torch_judge import check
check('kv_cache')


🧪 Testing: KV Cache Attention (Hard)
──────────────────────────────────────────────────
torch.Size([2, 8, 64]) torch.Size([2, 8, 64]) torch.Size([2, 8, 64])
torch.Size([2, 4, 8, 16]) torch.Size([2, 4, 8, 16]) torch.Size([2, 4, 8, 16])
torch.Size([2, 4, 8, 16])
torch.Size([2, 8, 64])
  ✅ [1/5] Output shape (no cache) (1.8ms)
torch.Size([2, 8, 64]) torch.Size([2, 8, 64]) torch.Size([2, 8, 64])
torch.Size([2, 4, 8, 16]) torch.Size([2, 4, 8, 16]) torch.Size([2, 4, 8, 16])
torch.Size([2, 4, 8, 16])
torch.Size([2, 8, 64])
  ✅ [2/5] Cache structure (0.7ms)
torch.Size([1, 4, 32]) torch.Size([1, 4, 32]) torch.Size([1, 4, 32])
torch.Size([1, 2, 4, 16]) torch.Size([1, 2, 4, 16]) torch.Size([1, 2, 4, 16])
torch.Size([1, 2, 4, 16])
torch.Size([1, 4, 32])
torch.Size([1, 1, 32]) torch.Size([1, 1, 32]) torch.Size([1, 1, 32])
torch.Size([1, 2, 1, 16]) torch.Size([1, 2, 1, 16]) torch.Size([1, 2, 1, 16])
After concatenation: torch.Size([1, 2, 5, 16]) torch.Size([1, 2, 5, 16])
torch.Size([1, 2, 1, 16])
t